**Cell 1 — Imports and configuration**

In [1]:
import os
import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import norm, t as tdist, kendalltau, rankdata
from scipy.optimize import minimize, minimize_scalar
from scipy.special import gammaln
from scipy.ndimage import uniform_filter1d

DATA_DIR = "../Data"

NU_FIXED = 4    # fixed df for t(4) copula, as in the reference table
NU_TCDF = 5     # df for bivariate t-cdf TDC row
EPS = 1e-6

WINDOW = 250
LEVEL = 0.05
N_SIM = 10_000
W = (1, 1)
COPULAS = ["Gauss", "t", "Gumbel", "Clayton", "Frank"]

**Cell 2 — Load data and compute standardized log-returns**

In [2]:
def load_close(filename):
    path = os.path.join(DATA_DIR, filename)
    s = pd.read_csv(path, parse_dates=["Date"]).set_index("Date")["Close"]
    s.index = pd.to_datetime(s.index, format="%d.%m.%Y")
    return s

def std_log_returns(s):
    lr = np.log(s / s.shift(1)).dropna()
    return (lr - lr.mean()) / lr.std()

r_bay = std_log_returns(load_close("BAYN_DE_price_volume.csv"))
r_sie = std_log_returns(load_close("SIE_DE_price_volume.csv"))
idx = r_bay.index.intersection(r_sie.index)
x = r_bay.loc[idx].values
y = r_sie.loc[idx].values

**Cell 3 — Fit t-margins and build pseudo-uniform observations**

In [3]:
def fit_t_margin(z, bounds=((2, 30), (-1, 1), (0.1, 5))):
    """Fit t-distribution to z via MLE; return (u=F_t(z), df, sigma)."""
    result = minimize(
        lambda p: -np.sum(tdist.logpdf(z, df=p[0], loc=p[1], scale=p[2])),
        x0=[4, 0, 1], bounds=bounds, method="L-BFGS-B",
    )
    df, loc, scale = result.x
    return tdist.cdf(z, df=df, loc=loc, scale=scale), df, scale

u, df_x, _ = fit_t_margin(x)
v, df_y, _ = fit_t_margin(y)
u = np.clip(u, EPS, 1 - EPS)
v = np.clip(v, EPS, 1 - EPS)

tau, _ = kendalltau(x, y)
rho_tau = np.sin(np.pi / 2 * tau)
print(f"Kendall tau = {tau:.4f},  implied rho = {rho_tau:.4f}")

Kendall tau = 0.3597,  implied rho = 0.5354


**Cell 4 — Copula log-likelihood functions**

In [4]:
def gauss_copula_loglik(rho, u, v):
    x_, y_ = norm.ppf(u), norm.ppf(v)
    r2 = rho ** 2
    ll = -0.5 * np.log(1 - r2) - (r2 * (x_**2 + y_**2) - 2 * rho * x_ * y_) / (2 * (1 - r2))
    return np.sum(ll)

def t_copula_loglik(rho, nu, u, v):
    x_, y_ = tdist.ppf(u, df=nu), tdist.ppf(v, df=nu)
    r2 = rho ** 2
    ll = (gammaln((nu + 2) / 2) + gammaln(nu / 2) - 2 * gammaln((nu + 1) / 2)
          - 0.5 * np.log(1 - r2)
          - ((nu + 2) / 2) * np.log(1 + (x_**2 + y_**2 - 2 * rho * x_ * y_) / (nu * (1 - r2)))
          + ((nu + 1) / 2) * (np.log(1 + x_**2 / nu) + np.log(1 + y_**2 / nu)))
    return np.sum(ll)

def gumbel_copula_loglik(theta, u, v):
    if theta < 1:
        return -np.inf
    lu, lv = -np.log(u), -np.log(v)
    A = (lu**theta + lv**theta) ** (1 / theta)
    ll = (-A + np.log(1 / u) + np.log(1 / v)
          + (1 / theta - 2) * np.log(lu**theta + lv**theta)
          + np.log(A + theta - 1) + (theta - 1) * (np.log(lu) + np.log(lv)))
    return np.sum(ll)

def clayton_copula_loglik(theta, u, v):
    if theta <= 0:
        return -np.inf
    ll = (np.log(1 + theta) - (1 + theta) * (np.log(u) + np.log(v))
          - (2 + 1 / theta) * np.log(u**(-theta) + v**(-theta) - 1))
    return np.sum(ll)

def frank_copula_loglik(theta, u, v):
    if abs(theta) < 1e-6:
        return -np.inf
    et = np.exp(-theta)
    num = -theta * (1 - et) * np.exp(-theta * (u + v))
    den = ((1 - et) - (1 - np.exp(-theta * u)) * (1 - np.exp(-theta * v))) ** 2
    return np.sum(np.log(np.maximum(num / den, 1e-300)))

**Cell 5 — Fit copula parameters via MLE**

In [5]:
def fit_copula(loglik_fn, bounds, *extra_args):
    result = minimize_scalar(
        lambda p: -loglik_fn(p, *extra_args, u, v), bounds=bounds, method="bounded"
    )
    return result.x

rho_gauss = fit_copula(lambda p, u, v: gauss_copula_loglik(p, u, v), (-0.99, 0.99))
rho_t = fit_copula(lambda p, u, v: t_copula_loglik(p, NU_FIXED, u, v), (-0.99, 0.99))
theta_gum = fit_copula(lambda p, u, v: gumbel_copula_loglik(p, u, v), (1.0, 10.0))
theta_cla = fit_copula(lambda p, u, v: clayton_copula_loglik(p, u, v), (0.01, 10.0))
theta_fra = fit_copula(lambda p, u, v: frank_copula_loglik(p, u, v), (0.01, 20.0))

**Cell 6 — TDC formulas**

In [6]:
def tdc_gauss():
    return 0.0

def tdc_t(rho, nu):
    x_ = np.sqrt((nu + 1) * (1 - rho) / (1 + rho))
    return 2 * tdist.sf(x_, df=nu + 1)

def tdc_gumbel_upper(theta):
    return 2 - 2 ** (1 / theta)

def tdc_clayton_lower(theta):
    return 2 ** (-1 / theta)

def tdc_frank():
    return 0.0

**Cell 7 — Nonparametric TDC estimator**

In [7]:
def msrnonp_tdc(a, b, bandwidth=20):
    T = len(a)
    s1, s2 = rankdata(a), rankdata(b)
    lam = np.array([np.sum((s1 > (T - k)) & (s2 > (T - k))) / k for k in range(1, T)])
    xs = uniform_filter1d(lam, size=bandwidth, mode="nearest")

    n = len(xs)
    bv = int(np.floor(0.005 * n))
    m = int(np.floor(np.sqrt(n - 2 * bv)))
    sd = np.std(xs[:n - 2 * bv], ddof=1)

    k_star = 0
    for k in range(n - m - 2 * bv + 1):
        if np.sum(np.abs(xs[k + 1:k + m] - xs[k])) <= sd:
            k_star = k
            break
    return float(np.mean(xs[k_star:k_star + m]))

def msrnonp_ltd(a, b, bandwidth=20):
    """Lower TDC: flip signs to reuse the upper-tail estimator."""
    return msrnonp_tdc(-a, -b, bandwidth)

np_upper = msrnonp_tdc(x, y)
np_lower = msrnonp_ltd(x, y)
tdc_tcdf = tdc_t(rho_gauss, NU_TCDF)

**Cell 8 — Build results table**

In [8]:
results = pd.DataFrame({
    "Copula": ["Gauss", "t(4)", "Gumbel", "Clayton", "Frank", "Nonparametric", "bivariate t-cdf"],
    "Parameters": [
        f"{rho_gauss:.4f}", f"{rho_t:.4f}", f"{theta_gum:.4f}",
        f"{theta_cla:.4f}", f"{theta_fra:.4f}", "-", str(NU_TCDF),
    ],
    "Upper TDC": [
        0, round(tdc_t(rho_t, NU_FIXED), 4), round(tdc_gumbel_upper(theta_gum), 4),
        0, 0, round(np_upper, 4), round(tdc_tcdf, 4),
    ],
    "Lower TDC": [
        0, round(tdc_t(rho_t, NU_FIXED), 4), 0,
        round(tdc_clayton_lower(theta_cla), 4), 0, round(np_lower, 4), round(tdc_tcdf, 4),
    ],
})

print(results.to_string(index=False))

         Copula Parameters  Upper TDC  Lower TDC
          Gauss     0.5132     0.0000     0.0000
           t(4)     0.5379     0.2749     0.2749
         Gumbel     1.5120     0.4184     0.0000
        Clayton     0.8379     0.0000     0.4373
          Frank    20.0000     0.0000     0.0000
  Nonparametric          -     0.3913     0.4209
bivariate t-cdf          5     0.2141     0.2141


**Cell 9 — Copula samplers for Monte Carlo**

In [9]:
def _clip_uv(u, v, eps=1e-9):
    return np.clip(u, eps, 1 - eps), np.clip(v, eps, 1 - eps)

def sample_gauss_copula(rho, n):
    cov = np.array([[1, rho], [rho, 1]])
    z = np.random.multivariate_normal([0, 0], cov, n)
    return norm.cdf(z[:, 0]), norm.cdf(z[:, 1])

def sample_t_copula(rho, nu, n):
    cov = np.array([[1, rho], [rho, 1]])
    z = np.random.multivariate_normal([0, 0], cov, n)
    chi2 = np.random.chisquare(nu, n)
    td = z / np.sqrt(chi2[:, None] / nu)
    return tdist.cdf(td[:, 0], df=nu), tdist.cdf(td[:, 1], df=nu)

def sample_gumbel_copula(theta, n):
    Vv = stats.levy_stable.rvs(1 / theta, 1, size=n, scale=(np.cos(np.pi / (2 * theta))) ** theta)
    E1 = np.random.exponential(1, n)
    E2 = np.random.exponential(1, n)
    return np.exp(-((E1 / Vv) ** (1 / theta))), np.exp(-((E2 / Vv) ** (1 / theta)))

def sample_clayton_copula(theta, n):
    u_ = np.random.uniform(0, 1, n)
    p = np.random.uniform(0, 1, n)
    v_ = u_ * (p ** (-theta / (1 + theta)) - 1 + u_ ** (-theta)) ** (-1 / theta)
    return _clip_uv(u_, v_)

def sample_frank_copula(theta, n):
    u_ = np.random.uniform(0, 1, n)
    p = np.random.uniform(0, 1, n)
    et = np.exp(-theta)
    etu = np.exp(-theta * u_)
    v_ = -1 / theta * np.log(1 + p * (et - 1) / (etu - p * (etu - 1)))
    return _clip_uv(u_, v_)

COPULA_FACTORIES = {
    "Gauss":   lambda s1, s2, d1, d2: (lambda n: sample_gauss_copula(rho_gauss, n)),
    "t":       lambda s1, s2, d1, d2: (lambda n: sample_t_copula(rho_t, NU_FIXED, n)),
    "Gumbel":  lambda s1, s2, d1, d2: (lambda n: sample_gumbel_copula(theta_gum, n)),
    "Clayton": lambda s1, s2, d1, d2: (lambda n: sample_clayton_copula(theta_cla, n)),
    "Frank":   lambda s1, s2, d1, d2: (lambda n: sample_frank_copula(theta_fra, n)),
}

**Cell 10 — Monte Carlo VaR and rolling backtest functions**

In [10]:
def mc_var_one_step(sampler_fn, sigma1, sigma2, df1, df2, n_sim=10_000, level=0.05, w=(1, 1)):
    u_s, v_s = sampler_fn(n_sim)
    r1 = tdist.ppf(u_s, df=df1) * sigma1
    r2 = tdist.ppf(v_s, df=df2) * sigma2
    pnl = w[0] * r1 + w[1] * r2
    return float(-np.quantile(pnl, level))

def fit_t_margin_window(z):
    """Re-fit t-margin on a rolling sub-window; return (sigma, df)."""
    _, df, sigma = fit_t_margin(z)
    return sigma, df

def backtest_var_fixed_copula(x_all, y_all, sampler_fn_factory, window=250, level=0.05, n_sim=10_000, w=(1, 1)):
    """Copula params fixed globally; only t-margins re-estimated per window."""
    T = len(x_all)
    violations, total = 0, 0

    for t in range(window, T):
        wx, wy = x_all[t - window:t], y_all[t - window:t]
        try:
            sig1, df1 = fit_t_margin_window(wx)
            sig2, df2 = fit_t_margin_window(wy)
        except Exception:
            continue

        sampler = sampler_fn_factory(sig1, sig2, df1, df2)
        var_hat = mc_var_one_step(sampler, sig1, sig2, df1, df2, n_sim=n_sim, level=level, w=w)

        realized = w[0] * x_all[t] + w[1] * y_all[t]
        violations += int(realized < -var_hat)
        total += 1

    return violations / total if total > 0 else float("nan")

def backtest_normal_var(x_all, y_all, window=250, level=0.05, w=(1, 1)):
    T = len(x_all)
    violations, total = 0, 0
    for t in range(window, T):
        port = w[0] * x_all[t - window:t] + w[1] * y_all[t - window:t]
        mu, sig = np.mean(port), np.std(port, ddof=1)
        var_hat = -(mu + norm.ppf(level) * sig)
        violations += int(w[0] * x_all[t] + w[1] * y_all[t] < -var_hat)
        total += 1
    return violations / total if total > 0 else float("nan")

**Cell 11 — Run backtests and print results table**

In [11]:
print(f"\nRunning rolling-window VaR backtest  (window={WINDOW}, level={LEVEL:.0%}, n_sim={N_SIM:,})")
print("Pair: BAY - SIE\n")

vr = {}
for cop in COPULAS:
    print(f"  [{cop:8s}]  ...", end=" ", flush=True)
    vr[cop] = backtest_var_fixed_copula(
        x, y, sampler_fn_factory=COPULA_FACTORIES[cop],
        window=WINDOW, level=LEVEL, n_sim=N_SIM, w=W,
    )
    print(f"violation rate = {vr[cop]:.4f}")

vr["Normal"] = backtest_normal_var(x, y, window=WINDOW, level=LEVEL, w=W)
print(f"  [Normal  ]  violation rate = {vr['Normal']:.4f}")

print("\n" + "-" * 42)
print(f"{'Copula':<22}  {'BAY-SIE':>10}")
print("-" * 42)
for cop in COPULAS:
    print(f"  {cop:<20}  {vr[cop]:>10.4f}")
print("-" * 42)
print(f"  {'Normal distribution':<20}  {vr['Normal']:>10.4f}")
print("-" * 42)
print(f"\nTarget violation rate: {LEVEL:.2f}")


Running rolling-window VaR backtest  (window=250, level=5%, n_sim=10,000)
Pair: BAY - SIE

  [Gauss   ]  ... violation rate = 0.0591
  [t       ]  ... violation rate = 0.0598
  [Gumbel  ]  ... violation rate = 0.0627
  [Clayton ]  ... violation rate = 0.0096
  [Frank   ]  ... violation rate = 0.0422
  [Normal  ]  violation rate = 0.0527

------------------------------------------
Copula                     BAY-SIE
------------------------------------------
  Gauss                     0.0591
  t                         0.0598
  Gumbel                    0.0627
  Clayton                   0.0096
  Frank                     0.0422
------------------------------------------
  Normal distribution       0.0527
------------------------------------------

Target violation rate: 0.05
